In [4]:
# Librerías base
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pyreadstat
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.style.use('ggplot')

In [5]:
# Carga del diccionario CIE-10
cie10 = pd.read_excel(
    "defunciones/def_variables.xlsx",
    sheet_name="CIE-10",
    header=None,
    skiprows=2,        # saltamos fila 1 (título) y fila 2 (encabezado)
    usecols="A,B",
    names=["codigo", "descripcion"]
)

print(f"Total de códigos CIE-10 cargados: {len(cie10)}")
print(f"\nPrimeros registros:")
cie10.head(10)

Total de códigos CIE-10 cargados: 14361

Primeros registros:


,codigo,descripcion
0,A00,Cólera
1,A000,"Cólera debido a Vibrio cholerae 01, biotipo ch..."
2,A001,"Cólera debido a Vibrio cholerae 01, biotipo el..."
3,A009,"Cólera, no especificado"
4,A01,Fiebres tifoidea y paratifoidea
5,A010,Fiebre tifoidea
6,A011,Fiebre paratifoidea A
7,A012,Fiebre paratifoidea B
8,A013,Fiebre paratifoidea C
9,A014,"Fiebre paratifoidea, no especificada"


In [6]:
# Auditoría del diccionario CIE-10
print("=== NULOS ===")
print(cie10.isnull().sum())

print(f"\n=== DUPLICADOS EN CÓDIGO ===")
print(f"Códigos duplicados: {cie10['codigo'].duplicated().sum()}")

print(f"\n=== FORMATO DE CÓDIGOS ===")
print(f"Longitud mínima: {cie10['codigo'].str.len().min()}")
print(f"Longitud máxima: {cie10['codigo'].str.len().max()}")
print(f"¿Contiene puntos?: {cie10['codigo'].str.contains('.', regex=False).sum()} códigos")
print(f"¿Contiene espacios?: {cie10['codigo'].str.contains(' ').sum()} códigos")

print(f"\n=== MUESTRA POR LONGITUD ===")
for l in sorted(cie10['codigo'].str.len().unique()):
    ejemplo = cie10[cie10['codigo'].str.len() == l]['codigo'].iloc[0]
    count = (cie10['codigo'].str.len() == l).sum()
    print(f"  Longitud {l}: {count:,} códigos  →  ejemplo: '{ejemplo}'")

=== NULOS ===
codigo         0
descripcion    0
dtype: int64

=== DUPLICADOS EN CÓDIGO ===
Códigos duplicados: 1

=== FORMATO DE CÓDIGOS ===
Longitud mínima: 3
Longitud máxima: 4
¿Contiene puntos?: 0 códigos
¿Contiene espacios?: 0 códigos

=== MUESTRA POR LONGITUD ===
  Longitud 3: 1,800 códigos  →  ejemplo: 'A00'
  Longitud 4: 12,561 códigos  →  ejemplo: 'A000'


In [7]:
# Identificar y eliminar duplicado
duplicado = cie10[cie10['codigo'].duplicated(keep=False)]
print("=== CÓDIGO DUPLICADO ===")
print(duplicado)

# Limpieza: quedarse con la primera ocurrencia
cie10 = cie10.drop_duplicates(subset='codigo', keep='first').reset_index(drop=True)
print(f"\nDiccionario limpio: {len(cie10):,} códigos únicos")

=== CÓDIGO DUPLICADO ===
      codigo                                        descripcion
14332   U129  Vacunas covid-19 que causan efectos adversos e...
14333   U129  Vacunas covid-19 que causan efectos adversos e...

Diccionario limpio: 14,360 códigos únicos


In [8]:
# Inspección de un año de defunciones para entender la estructura
df_muestra, meta = pyreadstat.read_sav("defunciones/def2022.sav")

print(f"Registros: {len(df_muestra):,}")
print(f"Variables: {df_muestra.shape[1]}")
print(f"\n=== COLUMNAS Y TIPOS ===")
print(df_muestra.dtypes)
print(f"\n=== PRIMERAS FILAS ===")
df_muestra.head()

Registros: 95,386
Variables: 27

=== COLUMNAS Y TIPOS ===
Depreg     float64
Mupreg         str
Mesreg     float64
Añoreg     float64
Depocu     float64
Mupocu         str
Sexo       float64
Diaocu     float64
Mesocu     float64
Añoocu     float64
Edadif     float64
Perdif     float64
Puedif     float64
Ecidif     float64
Escodif    float64
Ciuodif        str
Pnadif     float64
Dnadif     float64
Mnadif         str
Nacdif     float64
Predif     float64
Dredif     float64
Mredif         str
Caudef         str
Asist      float64
Ocur       float64
Cerdef     float64
dtype: object

=== PRIMERAS FILAS ===


,Depreg,Mupreg,Mesreg,Añoreg,Depocu,Mupocu,Sexo,Diaocu,Mesocu,Añoocu,Edadif,Perdif,Puedif,Ecidif,Escodif,Ciuodif,Pnadif,Dnadif,Mnadif,Nacdif,Predif,Dredif,Mredif,Caudef,Asist,Ocur,Cerdef
0,13.0,1301,5.0,2022.0,13.0,1301,1.0,17.0,5.0,2022.0,39.0,3.0,5.0,1.0,2.0,92,484.0,23.0,2300,484.0,320.0,13.0,1315,X482,1.0,1.0,1.0
1,13.0,1318,9.0,2022.0,13.0,1318,2.0,17.0,8.0,2022.0,33.0,3.0,5.0,1.0,1.0,97,484.0,23.0,2300,484.0,320.0,13.0,1318,O720,5.0,6.0,9.0
2,13.0,1301,6.0,2022.0,13.0,1301,1.0,2.0,6.0,2022.0,39.0,3.0,5.0,2.0,2.0,92,484.0,23.0,2300,484.0,320.0,13.0,1326,X482,1.0,1.0,1.0
3,13.0,1326,4.0,2022.0,13.0,1326,2.0,22.0,4.0,2022.0,35.0,3.0,5.0,1.0,1.0,97,484.0,23.0,2300,484.0,320.0,13.0,1326,W749,4.0,9.0,9.0
4,14.0,1420,2.0,2022.0,14.0,1420,1.0,17.0,1.0,2022.0,35.0,3.0,5.0,1.0,1.0,97,484.0,23.0,2300,484.0,320.0,14.0,1420,W849,9.0,9.0,9.0


In [9]:
# Auditoría de la columna Caudef
caudef = df_muestra['Caudef'].copy()

print(f"=== NULOS ===")
print(f"Nulos en Caudef: {caudef.isnull().sum():,} ({caudef.isnull().mean()*100:.1f}%)")

print(f"\n=== FORMATO ===")
caudef_notnull = caudef.dropna()
print(f"Longitud mínima: {caudef_notnull.str.len().min()}")
print(f"Longitud máxima: {caudef_notnull.str.len().max()}")
print(f"¿Contiene puntos?: {caudef_notnull.str.contains('.', regex=False).sum()} registros")
print(f"¿Contiene espacios?: {caudef_notnull.str.contains(' ').sum()} registros")
print(f"¿Contiene minúsculas?: {caudef_notnull.str.contains('[a-z]').sum()} registros")

print(f"\n=== MATCH CON DICCIONARIO CIE-10 ===")
codigos_cie = set(cie10['codigo'])
match = caudef_notnull.isin(codigos_cie).sum()
no_match = (~caudef_notnull.isin(codigos_cie)).sum()
print(f"Con match:    {match:,} ({match/len(caudef_notnull)*100:.1f}%)")
print(f"Sin match:    {no_match:,} ({no_match/len(caudef_notnull)*100:.1f}%)")

print(f"\n=== EJEMPLOS SIN MATCH ===")
print(caudef_notnull[~caudef_notnull.isin(codigos_cie)].value_counts().head(15))

=== NULOS ===
Nulos en Caudef: 0 (0.0%)

=== FORMATO ===
Longitud mínima: 4
Longitud máxima: 4
¿Contiene puntos?: 0 registros
¿Contiene espacios?: 0 registros
¿Contiene minúsculas?: 0 registros

=== MATCH CON DICCIONARIO CIE-10 ===
Con match:    95,386 (100.0%)
Sin match:    0 (0.0%)

=== EJEMPLOS SIN MATCH ===
Series([], Name: count, dtype: int64)


In [10]:
# Auditoría de variables demográficas clave
cols_clave = ['Depocu', 'Mupocu', 'Sexo', 'Diaocu', 'Mesocu', 'Añoocu', 'Edadif', 'Perdif']

print("=== NULOS POR VARIABLE ===")
print(df_muestra[cols_clave].isnull().sum().to_string())

print("\n=== ESTADÍSTICAS DESCRIPTIVAS ===")
print(df_muestra[cols_clave].describe().round(2).to_string())

print("\n=== VALORES ÚNICOS - VARIABLES CATEGÓRICAS ===")
for col in ['Sexo', 'Depocu', 'Perdif']:
    print(f"\n{col}: {sorted(df_muestra[col].dropna().unique().tolist())}")

=== NULOS POR VARIABLE ===
Depocu    0
Mupocu    0
Sexo      0
Diaocu    0
Mesocu    0
Añoocu    0
Edadif    0
Perdif    0

=== ESTADÍSTICAS DESCRIPTIVAS ===
         Depocu      Sexo    Diaocu    Mesocu   Añoocu    Edadif    Perdif
count  95386.00  95386.00  95386.00  95386.00  95386.0  95386.00  95386.00
mean       8.72      1.45     15.67      6.38   2022.0     60.44      2.91
std        6.65      0.50      8.77      3.54      0.0     54.25      0.51
min        1.00      1.00      1.00      1.00   2022.0      0.00      1.00
25%        1.00      1.00      8.00      3.00   2022.0     40.00      3.00
50%        9.00      1.00     16.00      6.00   2022.0     64.00      3.00
75%       14.00      2.00     23.00      9.00   2022.0     79.00      3.00
max       22.00      2.00     31.00     12.00   2022.0    999.00      9.00

=== VALORES ÚNICOS - VARIABLES CATEGÓRICAS ===

Sexo: [1.0, 2.0]

Depocu: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0

In [11]:
# Leer solo columna A de la hoja Defunciones para ver qué variables existen
hoja_def = pd.read_excel(
    "defunciones/def_variables.xlsx",
    sheet_name="Defunciones",
    header=None,
    usecols="A",
    names=["variable"]
)

# Filtrar solo filas que tienen nombre de variable (no vacías)
variables_doc = hoja_def['variable'].dropna().reset_index(drop=True)
print(f"Variables documentadas en el diccionario: {len(variables_doc)}")
print()
for i, v in enumerate(variables_doc):
    print(f"  {i+1:02d}. {v}")

Variables documentadas en el diccionario: 29

  01. Valores de las variables defunciones
  02. Valor
  03. Departamento de registro
  04. Municipio de registro
  05. Mes de registro
  06. Año de registro
  07. Departamento de ocurrencia
  08. Municipio de ocurrencia
  09. Sexo del difunto(a)
  10. Día de ocurrencia
  11. Mes de ocurrencia
  12. Año ocurrencia
  13. Edad del difunto(a)
  14. Periodo de edad del difunto(a)
  15. Pueblo de pertenencia del difunto(a)
  16. Estado civil del difunto(a)
  17. Escolaridad del difunto(a)
  18. Ocupación (Subgrupos CIUO-08) del difunto(a)
  19. Pais de nacimiento del difunto(a)
  20. Departamento de nacimiento del difunto(a)
  21. Municipio de nacimiento del difunto(a)
  22. Nacionalidad del difunto(a)
  23. Pais de residencia del difunto(a)
  24. Departamento de residencia del difunto(a)
  25. Municipio de residencia del difunto(a)
  26. Causa de defuncion
  27. Asistencia recibida
  28. Sitio de ocurrencia
  29. Quien certifica


In [12]:
# Carga de todos los años de defunciones (solo columnas necesarias)
COLS_DEF = ['Depocu', 'Mupocu', 'Sexo', 'Diaocu', 'Mesocu', 'Añoocu', 'Edadif', 'Perdif', 'Caudef']
AÑOS_DEF  = range(2011, 2023)

frames = []
for año in AÑOS_DEF:
    df_año, _ = pyreadstat.read_sav(
        f"defunciones/def{año}.sav",
        usecols=COLS_DEF
    )
    frames.append(df_año)
    print(f"  def{año}.sav → {len(df_año):,} registros")

defunciones = pd.concat(frames, ignore_index=True)
print(f"\nTotal consolidado: {len(defunciones):,} registros | {defunciones.shape[1]} variables")

  def2011.sav → 72,354 registros
  def2012.sav → 72,657 registros
  def2013.sav → 76,639 registros
  def2014.sav → 77,807 registros
  def2015.sav → 80,876 registros
  def2016.sav → 82,565 registros
  def2017.sav → 81,726 registros
  def2018.sav → 83,071 registros
  def2019.sav → 85,600 registros
  def2020.sav → 96,001 registros
  def2021.sav → 118,465 registros
  def2022.sav → 95,386 registros

Total consolidado: 1,023,147 registros | 9 variables


In [13]:
# Auditoría del dataset consolidado
print("=== NULOS POR VARIABLE ===")
nulos = defunciones.isnull().sum()
pct   = (nulos / len(defunciones) * 100).round(2)
print(pd.DataFrame({'Nulos': nulos, 'Porcentaje': pct}).to_string())

print(f"\n=== VALORES ESPECIALES ===")
print(f"Edadif = 999:  {(defunciones['Edadif'] == 999).sum():,}")
print(f"Sexo   = 9:    {(defunciones['Sexo']   == 9.0).sum():,}")
print(f"Depocu = 99:   {(defunciones['Depocu'] == 99).sum():,}")
print(f"Perdif únicos: {sorted(defunciones['Perdif'].dropna().unique().tolist())}")

print(f"\n=== RANGO DE AÑOS ===")
print(defunciones['Añoocu'].value_counts().sort_index().to_string())

=== NULOS POR VARIABLE ===
         Nulos  Porcentaje
Depocu       0        0.00
Sexo         0        0.00
Diaocu       0        0.00
Mesocu       0        0.00
Edadif       0        0.00
Perdif       0        0.00
Caudef       0        0.00
Mupocu   72354        7.07
Añoocu  299457       29.27

=== VALORES ESPECIALES ===
Edadif = 999:  7,022
Sexo   = 9:    0
Depocu = 99:   0
Perdif únicos: [1.0, 2.0, 3.0, 9.0]

=== RANGO DE AÑOS ===
Añoocu
2015.0     80876
2016.0     82565
2017.0     81726
2018.0     83071
2019.0     85600
2020.0     96001
2021.0    118465
2022.0     95386


In [14]:
# Inspeccionar columnas reales de los 4 años problemáticos
años_problematicos = [2011, 2012, 2013, 2014]

for año in años_problematicos:
    df_temp, meta = pyreadstat.read_sav(f"defunciones/def{año}.sav")
    print(f"\ndef{año}.sav — {df_temp.shape[1]} columnas:")
    print(list(df_temp.columns))


def2011.sav — 26 columnas:
['Depreg', 'mupreg', 'Mesreg', 'Añoreg', 'Depocu', 'mupocu', 'Areag', 'Sexo', 'Diaocu', 'Mesocu', 'añoocu', 'Edadif', 'Perdif', 'Getdif', 'Ecidif', 'Escodif', 'Ocudif', 'Dnadif', 'Mnadif', 'Nacdif', 'Dredif', 'Mredif', 'Caudef', 'Asist', 'Ocur', 'Cerdef']

def2012.sav — 27 columnas:
['Depreg', 'Mupreg', 'Mesreg', 'Añoreg', 'Depocu', 'Mupocu', 'Areag', 'Sexo', 'Diaocu', 'Mesocu', 'Edadif', 'Perdif', 'Getdif', 'Ecidif', 'Escodif', 'Ocudif', 'Pnadif', 'Dnadif', 'Mnadif', 'Nacdif', 'Predif', 'Dredif', 'Mredif', 'Caudef', 'Asist', 'Ocur', 'Cerdef']

def2013.sav — 28 columnas:
['Depreg', 'Mupreg', 'Mesreg', 'Añoreg', 'Depocu', 'Mupocu', 'Areag', 'Sexo', 'Diaocu', 'Mesocu', 'Edadif', 'Perdif', 'Puedif', 'Ecidif', 'Escodif', 'Ciuodif', 'Pnadif', 'Dnadif', 'Mnadif', 'Nacdif', 'Predif', 'Dredif', 'Mredif', 'Caudef', 'caudef.descrip', 'Asist', 'Ocur', 'Cerdef']

def2014.sav — 27 columnas:
['Depreg', 'Mupreg', 'Mesreg', 'Añoreg', 'Depocu', 'Mupocu', 'Areag', 'Sexo', 'Di

In [15]:
# Recarga robusta manejando inconsistencias de naming entre años
COLS_TARGET = {
    'depocu'  : 'depocu',
    'mupocu'  : 'mupocu',
    'sexo'    : 'sexo',
    'diaocu'  : 'diaocu',
    'mesocu'  : 'mesocu',
    'edadif'  : 'edadif',
    'perdif'  : 'perdif',
    'caudef'  : 'caudef',
    'añoocu'  : 'anio',   # 2011 lo tiene con este nombre
}

frames = []
for año in range(2011, 2023):
    df_temp, _ = pyreadstat.read_sav(f"defunciones/def{año}.sav")

    # Normalizar todos los nombres a minúscula
    df_temp.columns = df_temp.columns.str.lower()

    # Renombrar al esquema estándar
    df_temp = df_temp.rename(columns=COLS_TARGET)

    # Si el año no vino en el archivo, inyectarlo
    if 'anio' not in df_temp.columns:
        df_temp['anio'] = año

    # Seleccionar solo columnas que necesitamos
    cols_finales = ['anio', 'depocu', 'mupocu', 'sexo', 'diaocu', 'mesocu', 'edadif', 'perdif', 'caudef']
    df_temp = df_temp[cols_finales]

    frames.append(df_temp)
    print(f"  def{año}.sav → {len(df_temp):,} registros | anio: {df_temp['anio'].iloc[0]:.0f}")

defunciones = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(defunciones):,} registros | {defunciones.shape[1]} variables")
print(f"Nulos en anio: {defunciones['anio'].isnull().sum()}")
print(f"\nDistribución por año:")
print(defunciones['anio'].value_counts().sort_index().to_string())

  def2011.sav → 72,354 registros | anio: 2011
  def2012.sav → 72,657 registros | anio: 2012
  def2013.sav → 76,639 registros | anio: 2013
  def2014.sav → 77,807 registros | anio: 2014
  def2015.sav → 80,876 registros | anio: 2015
  def2016.sav → 82,565 registros | anio: 2016
  def2017.sav → 81,726 registros | anio: 2017
  def2018.sav → 83,071 registros | anio: 2018
  def2019.sav → 85,600 registros | anio: 2019
  def2020.sav → 96,001 registros | anio: 2020
  def2021.sav → 118,465 registros | anio: 2021
  def2022.sav → 95,386 registros | anio: 2022

Total: 1,023,147 registros | 9 variables
Nulos en anio: 0

Distribución por año:
anio
2011.0     72354
2012.0     72657
2013.0     76639
2014.0     77807
2015.0     80876
2016.0     82565
2017.0     81726
2018.0     83071
2019.0     85600
2020.0     96001
2021.0    118465
2022.0     95386


In [19]:
# Inspección de columnas reales por año
AÑOS_NEC = [2011, 2012, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

for año in AÑOS_NEC:
    df_temp, _ = pyreadstat.read_sav(f"necropsias/nec{año}.sav")
    df_temp.columns = df_temp.columns.str.lower()
    print(f"\nnec{año}.sav — {df_temp.shape[1]} columnas:")
    print(list(df_temp.columns))


nec2011.sav — 7 columnas:
['num_corre', 'mes_ocu', 'depto_ocu', 'edad_person', 'g_edad', 'sexo_person', 'causa_muerte']

nec2012.sav — 7 columnas:
['num_corre', 'mes_ocu', 'depto_ocu', 'edad_person', 'g_edad', 'sexo_person', 'causa_muerte']

nec2016.sav — 13 columnas:
['no_corrre', 'año_ocu', 'mes_ocu', 'diasem_ocu', 'dia_ocu', 'depto_ocu', 'edad_per', 'sexo_per', 'causa_muerte', 'g_edad_80ymás', 'g_edad_60ymás', 'edad_quinquenales', 'mayor_menor']

nec2017.sav — 14 columnas:
['núm_corre', 'año_ocu', 'mes_ocu', 'día_ocu', 'día_sem_ocu', 'depto_ocu', 'mupio_ocu', 'sexo_per_eva', 'edad_per', 'g_edad_60ymás', 'g_edad_80ymás', 'edad_quinquenales', 'menor_mayor', 'eva_mn']

nec2018.sav — 14 columnas:
['núm_corre', 'año_ocu', 'mes_ocu', 'día_ocu', 'dia_sem_ocu', 'depto_ocu', 'mupio_ocu', 'edad_per', 'g_edad_60ymás', 'g_edad_80ymás', 'edad_quinquenales', 'menor_mayor', 'sexo_per_eva', 'eva_mn']

nec2019.sav — 14 columnas:
['núm_corre', 'año_ocu', 'mes_ocu', 'día_ocu', 'día_sem_ocu', 'depto_o

In [20]:
for año in [2017, 2018]:
    df_temp, _ = pyreadstat.read_sav(f"necropsias/nec{año}.sav")
    df_temp.columns = df_temp.columns.str.lower()
    print(f"\nnec{año} — eva_mn únicos ({df_temp['eva_mn'].nunique()}):")
    print(df_temp['eva_mn'].value_counts().head(20))


nec2017 — eva_mn únicos (42):
eva_mn
20.0    3885
37.0    3489
2.0     1317
21.0     488
19.0     473
27.0     357
22.0     313
23.0     301
7.0      171
18.0     160
29.0     147
33.0     132
5.0       89
32.0      81
16.0      70
38.0      58
8.0       39
3.0       32
11.0      29
31.0      28
Name: count, dtype: int64

nec2018 — eva_mn únicos (39):
eva_mn
20.0    3491
37.0    3359
2.0     1347
21.0     560
27.0     416
19.0     408
22.0     391
23.0     248
18.0     229
29.0     165
7.0      158
33.0     119
5.0      102
16.0      77
38.0      77
32.0      53
8.0       47
31.0      31
4.0       24
11.0      21
Name: count, dtype: int64


In [21]:
hoja_nec = pd.read_excel(
    "necropsias/nec2018.xlsx",
    header=None,
    usecols="A",
    names=["variable"]
)

variables_nec = hoja_nec['variable'].dropna().reset_index(drop=True)
print(f"Variables documentadas: {len(variables_nec)}")
for i, v in enumerate(variables_nec):
    print(f"  {i+1:02d}. {v}")

Variables documentadas: 17
  01. NECROPSIAS
  02. Valores de las variables
  03. Valor
  04. NÚMERO DE CORRELATIVO
  05. AÑO DE OCURRENCIA
  06. MES DE OCURRENCIA
  07. DIA DE OCURRENCIA
  08. DIA DE LA SEMANA DE OCURRENCIA
  09. DEPARTAMENTO DE OCURRENCIA
  10. MUNICIPIO DE OCURRENCIA
  11. SEXO DE LA PERSONA EVALUADA
  12. EDAD SIMPLE
  13. GRUPO DE EDAD QUINQUENAL (MENOR 15 HASTA 60 Y MÁS)
  14. GRUPO DE EDAD QUINQUENAL (MENOR 15 HASTA 80 Y MÁS)
  15. EDADES QUINQUENALES (A PARTIR DE 0 HASTA 80 Y MÁS)
  16. MENOR O MAYOR DE EDAD
  17. EVALUACIÓN MÉDICO DE NECROPSIA


In [22]:
# Leer diccionario de Evaluación Médico de Necropsia
dic_eva_mn = pd.read_excel(
    "necropsias/nec2018.xlsx",
    header=None,
    skiprows=446,        # empieza en fila 447 (índice 446)
    usecols="B,C",
    names=["codigo", "descripcion"]
).dropna().reset_index(drop=True)

print(f"Total de códigos eva_mn: {len(dic_eva_mn)}")
print(dic_eva_mn.to_string())

Total de códigos eva_mn: 42
    codigo                                                                                descripcion
0        1                                                                                 Amputación
1        2                                                                                    Asfixia
2        3                                                                             Cardiomiopatía
3        4                                                                               Decapitación
4        5                                                                                  Depleción
5        6                                                                               Desnutrición
6        7                                                                             Edema pulmonar
7        8                                                                              Electrocución
8        9                                            

In [25]:
print(dic_eva_mn[dic_eva_mn['codigo'].between(24, 39)].to_string())

    codigo                         descripcion
23      24           Malformaciones congénitas
24      25                          Meningitis
25      26  Mordedura y/o picadura de insectos
26      27                            Neumonía
27      28    Otras enfermedades al pericardio
28      29                        Pancreatitis
29      30                         Peritonitis
30      31                          Prematurez
31      32                           Quemadura
32      33                              Sepsis
33      35               Taponamiento cardiaco
34      36             Transtornos metabólicos
35      37                         Traumatismo
36      38                       Tromboembolia
37      39                           Trombosis


In [26]:
# Ver qué valores tiene causa_muerte en años que no usan eva_mn
for año in [2011, 2012, 2016, 2019, 2020, 2021, 2022, 2023, 2024]:
    df_temp, _ = pyreadstat.read_sav(f"necropsias/nec{año}.sav")
    df_temp.columns = df_temp.columns.str.lower()
    causa = df_temp['causa_muerte']
    print(f"\nnec{año} — causa_muerte | tipo: {causa.dtype} | únicos: {causa.nunique()}")
    print(causa.value_counts().head(5))


nec2011 — causa_muerte | tipo: float64 | únicos: 6
causa_muerte
1.0    5216
2.0    2995
6.0    2607
3.0    1274
5.0     569
Name: count, dtype: int64

nec2012 — causa_muerte | tipo: float64 | únicos: 6
causa_muerte
1.0    5060
2.0    3080
6.0    2795
3.0     846
4.0     586
Name: count, dtype: int64

nec2016 — causa_muerte | tipo: float64 | únicos: 41
causa_muerte
20.0    4012
37.0    3859
2.0     1221
19.0     527
21.0     520
Name: count, dtype: int64

nec2019 — causa_muerte | tipo: float64 | únicos: 39
causa_muerte
37.0    3325
20.0    3226
2.0     1349
21.0     790
19.0     401
Name: count, dtype: int64

nec2020 — causa_muerte | tipo: float64 | únicos: 40
causa_muerte
37.0    2900
20.0    2278
2.0     1059
21.0     637
19.0     359
Name: count, dtype: int64

nec2021 — causa_muerte | tipo: float64 | únicos: 33
causa_muerte
37.0    3722
20.0    2655
2.0     1280
21.0     714
19.0     383
Name: count, dtype: int64

nec2022 — causa_muerte | tipo: float64 | únicos: 33
causa_muerte
37.0

In [27]:
# Ver si el .sav de 2011 tiene etiquetas embebidas para causa_muerte
df_temp, meta = pyreadstat.read_sav("necropsias/nec2011.sav")
df_temp.columns = df_temp.columns.str.lower()
print("Etiquetas embebidas en causa_muerte:")
print(meta.value_labels)

Etiquetas embebidas en causa_muerte:
{'labels0': {1.0: 'Enero', 2.0: 'Febrero', 3.0: 'Marzo', 4.0: 'Abril', 5.0: 'Mayo', 6.0: 'Junio', 7.0: 'Julio', 8.0: 'Agosto', 9.0: 'Septiembre', 10.0: 'Octubre', 11.0: 'Noviembre', 12.0: 'Diciembre'}, 'labels1': {1.0: 'Guatemala', 2.0: 'El Progreso', 3.0: 'Sacatepequez', 4.0: 'Chimaltenango', 5.0: 'Escuintla', 6.0: 'Santa Rosa', 7.0: 'Solola', 8.0: 'Totonicapan', 9.0: 'Quetzaltenango', 10.0: 'Suchitepequez', 11.0: 'Retalhuleu', 12.0: 'San Marcos', 13.0: 'Huehuetenango', 14.0: 'Quiche', 15.0: 'Baja Verapaz', 16.0: 'Alta Verapaz', 17.0: 'Peten', 18.0: 'Izabal', 19.0: 'Zacapa', 20.0: 'Chiquimulal', 21.0: 'Jalapa', 22.0: 'Jutiapa'}, 'labels2': {999.0: 'Ignorado'}, 'labels3': {1.0: 'Menor de 15', 2.0: '15-19', 3.0: '20-24', 4.0: '25-29', 5.0: '30-34', 6.0: '35-39', 7.0: '40-44', 8.0: '45-49', 9.0: '50-54', 10.0: '55 y más', 11.0: 'Ignorado'}, 'labels4': {1.0: 'Hombre', 2.0: 'Mujer', 9.0: 'Ignorado'}, 'labels5': {1.0: 'Herida de arma de fuego', 2.0: 'Tra

In [28]:
# Diccionario sistema simple (2011-2012)
CAUSA_SIMPLE = {
    1.0: 0,  # Herida de arma de fuego  → No natural
    2.0: 0,  # Traumatismo              → No natural
    3.0: 0,  # Asfixia                  → No natural
    4.0: 0,  # Herida de arma blanca    → No natural
    5.0: None,  # Edema cerebral/pulmonar → Eliminar
    6.0: None,  # Otros                   → Eliminar
}

# Diccionario sistema completo (2016+)
CAUSA_COMPLETA = {
    1.0:  0,     # Amputación
    2.0:  0,     # Asfixia
    3.0:  1,     # Cardiomiopatía
    4.0:  0,     # Decapitación
    5.0:  None,  # Depleción             → Eliminar
    6.0:  1,     # Desnutrición
    7.0:  1,     # Edema pulmonar
    8.0:  0,     # Electrocución
    9.0:  1,     # Embolia cerebral
    10.0: 1,     # Embolia pulmonar
    11.0: 1,     # Enfermedad vascular
    12.0: 1,     # Enterocolitis
    13.0: None,  # Envenenamiento serpiente → Eliminar
    14.0: 1,     # Evento cerebrovascular
    15.0: 1,     # Evento vascular/aneurisma
    16.0: 1,     # Fibrosis y cirrosis
    17.0: 0,     # Fulguración
    18.0: 0,     # Hemorragias
    19.0: 0,     # Herida de arma blanca
    20.0: 0,     # Herida de arma de fuego
    21.0: None,  # Indeterminada         → Eliminar
    22.0: 1,     # Infarto
    23.0: 0,     # Intoxicación
    24.0: 1,     # Malformaciones congénitas
    25.0: 1,     # Meningitis
    26.0: None,  # Mordedura insectos    → Eliminar
    27.0: 1,     # Neumonía
    28.0: 1,     # Otras enf. pericardio
    29.0: 1,     # Pancreatitis
    30.0: 1,     # Peritonitis
    31.0: 1,     # Prematurez
    32.0: 0,     # Quemadura
    33.0: 1,     # Sepsis
    35.0: 1,     # Taponamiento cardiaco
    36.0: 1,     # Trastornos metabólicos
    37.0: 0,     # Traumatismo
    38.0: 1,     # Tromboembolia
    39.0: 1,     # Trombosis
    40.0: 1,     # Tuberculosis
    41.0: 1,     # Tumor maligno/neoplasia
    42.0: 1,     # Úlcera gástrica perforada
    43.0: 1,     # Embolia grasa
}

print("Diccionarios definidos.")
print(f"Sistema simple  : {len(CAUSA_SIMPLE)} códigos")
print(f"Sistema completo: {len(CAUSA_COMPLETA)} códigos")
print(f"\nNo naturales (0) en sistema completo : {sum(1 for v in CAUSA_COMPLETA.values() if v == 0)}")
print(f"Naturales    (1) en sistema completo : {sum(1 for v in CAUSA_COMPLETA.values() if v == 1)}")
print(f"Eliminados  (None) en sistema completo: {sum(1 for v in CAUSA_COMPLETA.values() if v is None)}")

Diccionarios definidos.
Sistema simple  : 6 códigos
Sistema completo: 42 códigos

No naturales (0) en sistema completo : 11
Naturales    (1) en sistema completo : 27
Eliminados  (None) en sistema completo: 4


In [29]:
# Mapeo de nombres de columnas por año hacia esquema estándar
COLS_NEC_MAP = {
    # año
    'año_ocu': 'anio', 'año_ing': 'anio',
    # mes
    'mes_ocu': 'mes', 'mes_ing': 'mes',
    # dia
    'dia_ocu': 'dia', 'día_ocu': 'dia', 'día_ing': 'dia',
    # departamento
    'depto_ocu': 'depto',
    # municipio
    'mupio_ocu': 'mupio',
    # edad
    'edad_person': 'edad', 'edad_per': 'edad',
    # sexo
    'sexo_person': 'sexo', 'sexo_per': 'sexo', 'sexo_per_eva': 'sexo',
    # causa
    'causa_muerte': 'causa', 'eva_mn': 'causa',
}

AÑOS_NEC = [2011, 2012, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
COLS_FINALES = ['anio', 'mes', 'dia', 'depto', 'mupio', 'edad', 'sexo', 'causa']

frames = []
for año in AÑOS_NEC:
    df_temp, _ = pyreadstat.read_sav(f"necropsias/nec{año}.sav")
    df_temp.columns = df_temp.columns.str.lower()
    df_temp = df_temp.rename(columns=COLS_NEC_MAP)

    # Inyectar año si no vino en el archivo
    if 'anio' not in df_temp.columns:
        df_temp['anio'] = año

    # Seleccionar solo columnas disponibles del set final
    cols_disponibles = [c for c in COLS_FINALES if c in df_temp.columns]
    df_temp = df_temp[cols_disponibles].copy()

    # Agregar columnas faltantes como NaN
    for col in COLS_FINALES:
        if col not in df_temp.columns:
            df_temp[col] = np.nan

    df_temp = df_temp[COLS_FINALES]
    frames.append(df_temp)
    print(f"  nec{año}.sav → {len(df_temp):,} registros")

necropsias = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(necropsias):,} registros | {necropsias.shape[1]} variables")
print(f"\nDistribución por año:")
print(necropsias['anio'].value_counts().sort_index().to_string())

  nec2011.sav → 13,137 registros
  nec2012.sav → 12,753 registros
  nec2016.sav → 12,179 registros
  nec2017.sav → 11,848 registros
  nec2018.sav → 11,512 registros
  nec2019.sav → 11,351 registros
  nec2020.sav → 8,819 registros
  nec2021.sav → 10,281 registros
  nec2022.sav → 10,729 registros
  nec2023.sav → 11,038 registros
  nec2024.sav → 2,598 registros

Total: 116,245 registros | 8 variables

Distribución por año:
anio
2011.0    13137
2012.0    12753
2016.0    12179
2017.0    11848
2018.0    11512
2019.0    11351
2020.0     8819
2021.0    10281
2022.0    10729
2023.0    11038
2024.0     2598


In [31]:
# Aplicar diccionario correcto según el año
def clasificar_causa_nec(row):
    if row['anio'] in [2011.0, 2012.0]:
        return CAUSA_SIMPLE.get(row['causa'], np.nan)
    else:
        return CAUSA_COMPLETA.get(row['causa'], np.nan)

necropsias['es_muerte_natural'] = necropsias.apply(clasificar_causa_nec, axis=1)

# Reporte antes de eliminar
total = len(necropsias)
naturales   = (necropsias['es_muerte_natural'] == 1).sum()
no_naturales = (necropsias['es_muerte_natural'] == 0).sum()
eliminados  = necropsias['es_muerte_natural'].isna().sum()

print("=== CLASIFICACIÓN DE NECROPSIAS ===")
print(f"Total original    : {total:,}")
print(f"Natural      (1)  : {naturales:,}  ({naturales/total*100:.1f}%)")
print(f"No natural   (0)  : {no_naturales:,}  ({no_naturales/total*100:.1f}%)")
print(f"Eliminados (None) : {eliminados:,}  ({eliminados/total*100:.1f}%)")

# Eliminar ambiguos
necropsias = necropsias[necropsias['es_muerte_natural'].notna()].reset_index(drop=True)
print(f"\nTotal después de limpieza: {len(necropsias):,}")
print(f"\nBalance final necropsias:")
print(necropsias['es_muerte_natural'].value_counts())

=== CLASIFICACIÓN DE NECROPSIAS ===
Total original    : 104,103
Natural      (1)  : 10,416  (10.0%)
No natural   (0)  : 93,687  (90.0%)
Eliminados (None) : 0  (0.0%)

Total después de limpieza: 104,103

Balance final necropsias:
es_muerte_natural
0    93687
1    10416
Name: count, dtype: int64


In [32]:
# Diagnóstico de registros faltantes
print(f"Total en necropsias actual: {len(necropsias):,}")
print(f"\nNulos por columna:")
print(necropsias.isnull().sum())
print(f"\nRegistros con causa NaN:")
print((necropsias['causa'].isna()).sum())
print(f"\nDistribución por año actual:")
print(necropsias['anio'].value_counts().sort_index().to_string())

Total en necropsias actual: 104,103

Nulos por columna:
anio                     0
mes                      0
dia                  19533
depto                    0
mupio                31126
edad                     0
sexo                     0
causa                    0
es_muerte_natural        0
dtype: int64

Registros con causa NaN:
0

Distribución por año actual:
anio
2011.0     9961
2012.0     9572
2016.0    11593
2017.0    11269
2018.0    10849
2019.0    10483
2020.0     8153
2021.0     9564
2022.0     9961
2023.0    10273
2024.0     2425


In [33]:
# Verificación de que los 12,142 eliminados tienen sentido
print(f"Registros originales (Celda 14): 116,245")
print(f"Registros actuales             : {len(necropsias):,}")
print(f"Diferencia eliminada           : {116245 - len(necropsias):,}")

# Cuántos tenía cada código eliminado
print(f"\n=== CÓDIGOS ELIMINADOS POR AÑO ===")
print(f"2011/2012 - código 5 (Edema) + 6 (Otros):")
print(f"  2011: 13,137 - 9,961 = {13137 - 9961:,}")
print(f"  2012: 12,753 - 9,572 = {12753 - 9572:,}")
print(f"\n2016+ - códigos 5,13,21,26:")
print(f"  Total eliminados 2016+: {(116245-104103)-(3176+3181):,}")

print(f"\n=== BALANCE FINAL NECROPSIAS ===")
print(necropsias['es_muerte_natural'].value_counts())
print(f"\nPorcentaje natural    : {(necropsias['es_muerte_natural']==1).mean()*100:.1f}%")
print(f"Porcentaje no natural : {(necropsias['es_muerte_natural']==0).mean()*100:.1f}%")

Registros originales (Celda 14): 116,245
Registros actuales             : 104,103
Diferencia eliminada           : 12,142

=== CÓDIGOS ELIMINADOS POR AÑO ===
2011/2012 - código 5 (Edema) + 6 (Otros):
  2011: 13,137 - 9,961 = 3,176
  2012: 12,753 - 9,572 = 3,181

2016+ - códigos 5,13,21,26:
  Total eliminados 2016+: 5,785

=== BALANCE FINAL NECROPSIAS ===
es_muerte_natural
0    93687
1    10416
Name: count, dtype: int64

Porcentaje natural    : 10.0%
Porcentaje no natural : 90.0%


In [34]:
# Clasificar defunciones por capítulo CIE-10
def clasificar_causa_def(codigo):
    if pd.isna(codigo):
        return np.nan
    cap = str(codigo)[0].upper()
    if cap in ['V', 'W', 'X', 'Y']:  # Causas externas → eliminar
        return np.nan
    if cap == 'Z':                    # Factores de salud → eliminar
        return np.nan
    return 1                          # Todo lo demás → Natural

defunciones['es_muerte_natural'] = defunciones['caudef'].apply(clasificar_causa_def)

total = len(defunciones)
naturales  = (defunciones['es_muerte_natural'] == 1).sum()
eliminados = defunciones['es_muerte_natural'].isna().sum()

print("=== CLASIFICACIÓN DE DEFUNCIONES ===")
print(f"Total original  : {total:,}")
print(f"Natural    (1)  : {naturales:,}  ({naturales/total*100:.1f}%)")
print(f"Eliminados      : {eliminados:,}  ({eliminados/total*100:.1f}%)")

# Eliminar no naturales y Z
defunciones = defunciones[defunciones['es_muerte_natural'].notna()].reset_index(drop=True)
print(f"\nTotal después de limpieza: {len(defunciones):,}")

print(f"\n=== CAPÍTULOS ELIMINADOS ===")
caps_eliminados = defunciones_backup = pd.concat(frames, ignore_index=True)
caps_eliminados = caps_eliminados[caps_eliminados['caudef'].notna()]
caps_eliminados = caps_eliminados[caps_eliminados['caudef'].str[0].str.upper().isin(['V','W','X','Y','Z'])]
print(caps_eliminados['caudef'].str[0].str.upper().value_counts())

=== CLASIFICACIÓN DE DEFUNCIONES ===
Total original  : 1,023,147
Natural    (1)  : 882,680  (86.3%)
Eliminados      : 140,467  (13.7%)

Total después de limpieza: 882,680

=== CAPÍTULOS ELIMINADOS ===


KeyError: 'caudef'

In [35]:
# Reporte de capítulos eliminados desde el total original
print("=== CAPÍTULOS ELIMINADOS (sobre total original) ===")
print(f"Total original    : 1,023,147")
print(f"Naturales (A-U)   : {len(defunciones):,}")
print(f"Eliminados V/W/X/Y/Z: {1023147 - len(defunciones):,}")
print(f"\n=== DEFUNCIONES LISTAS ===")
print(f"Total natural (1) : {len(defunciones):,}")
print(f"\nDistribución por año:")
print(defunciones['anio'].value_counts().sort_index().to_string())

=== CAPÍTULOS ELIMINADOS (sobre total original) ===
Total original    : 1,023,147
Naturales (A-U)   : 882,680
Eliminados V/W/X/Y/Z: 140,467

=== DEFUNCIONES LISTAS ===
Total natural (1) : 882,680

Distribución por año:
anio
2011.0     59763
2012.0     60341
2013.0     64033
2014.0     65396
2015.0     68617
2016.0     70463
2017.0     69977
2018.0     71489
2019.0     74421
2020.0     86624
2021.0    107454
2022.0     84102


In [36]:
# Esquema estándar final
# Necropsias:  anio, mes, dia, depto, mupio, edad, sexo, es_muerte_natural
# Defunciones: anio, mesocu, diaocu, depocu, mupocu, edadif, sexo, es_muerte_natural

# Renombrar defunciones al esquema de necropsias
defunciones_final = defunciones.rename(columns={
    'mesocu': 'mes',
    'diaocu': 'dia',
    'depocu': 'depto',
    'mupocu': 'mupio',
    'edadif': 'edad',
}).copy()

# Seleccionar solo columnas del esquema final
COLS_FINALES = ['anio', 'mes', 'dia', 'depto', 'mupio', 'edad', 'sexo', 'es_muerte_natural']

defunciones_final = defunciones_final[COLS_FINALES]
necropsias_final  = necropsias[COLS_FINALES]

# Unir ambos datasets
df = pd.concat([necropsias_final, defunciones_final], ignore_index=True)

print("=== DATASET UNIFICADO ===")
print(f"Total registros : {len(df):,}")
print(f"\nBalance de clases:")
print(df['es_muerte_natural'].value_counts())
print(f"\nPorcentaje:")
print(df['es_muerte_natural'].value_counts(normalize=True).mul(100).round(1))
print(f"\nColumnas: {list(df.columns)}")
print(f"\nNulos por columna:")
print(df.isnull().sum())

=== DATASET UNIFICADO ===
Total registros : 986,783

Balance de clases:
es_muerte_natural
1.0    893096
0.0     93687
Name: count, dtype: int64

Porcentaje:
es_muerte_natural
1.0    90.5
0.0     9.5
Name: proportion, dtype: float64

Columnas: ['anio', 'mes', 'dia', 'depto', 'mupio', 'edad', 'sexo', 'es_muerte_natural']

Nulos por columna:
anio                     0
mes                      0
dia                  19533
depto                    0
mupio                31126
edad                     0
sexo                     0
es_muerte_natural        0
dtype: int64


In [38]:
# Tratar valores especiales
df['edad'] = df['edad'].replace(999.0, np.nan)
df['sexo'] = df['sexo'].replace(9.0,   np.nan)

# Reporte de nulos completo post-limpieza
print("=== NULOS DESPUÉS DE TRATAR VALORES ESPECIALES ===")
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(2)
print(pd.DataFrame({'Nulos': nulos, 'Porcentaje': pct}).to_string())


=== NULOS DESPUÉS DE TRATAR VALORES ESPECIALES ===
                   Nulos  Porcentaje
anio                   0        0.00
mes                    0        0.00
dia                19533        1.98
depto                  0        0.00
mupio              31126        3.15
edad                6358        0.64
sexo                 214        0.02
es_muerte_natural      0        0.00


In [39]:
# Eliminar columna mupio
df = df.drop(columns=['mupio'])

# Imputar dia con mediana por mes
df['dia'] = df.groupby('mes')['dia'].transform(
    lambda x: x.fillna(x.median())
)

# Eliminar filas con nulos en variables críticas
df = df.dropna(subset=['edad', 'sexo']).reset_index(drop=True)

print("=== DATASET FINAL LIMPIO ===")
print(f"Total registros : {len(df):,}")
print(f"Columnas        : {list(df.columns)}")
print(f"\nNulos restantes:")
print(df.isnull().sum())
print(f"\nBalance de clases:")
print(df['es_muerte_natural'].value_counts())
print(df['es_muerte_natural'].value_counts(normalize=True).mul(100).round(1))

=== DATASET FINAL LIMPIO ===
Total registros : 980,303
Columnas        : ['anio', 'mes', 'dia', 'depto', 'edad', 'sexo', 'es_muerte_natural']

Nulos restantes:
anio                 0
mes                  0
dia                  0
depto                0
edad                 0
sexo                 0
es_muerte_natural    0
dtype: int64

Balance de clases:
es_muerte_natural
1.0    887468
0.0     92835
Name: count, dtype: int64
es_muerte_natural
1.0    90.5
0.0     9.5
Name: proportion, dtype: float64


In [40]:
# Verificar distribución de Perdif en defunciones original
# (antes de que lo perdiéramos al hacer el join)
df_perdif_check, _ = pyreadstat.read_sav("defunciones/def2022.sav")
df_perdif_check.columns = df_perdif_check.columns.str.lower()

print("=== DISTRIBUCIÓN DE PERDIF (muestra 2022) ===")
print(df_perdif_check['perdif'].value_counts().sort_index())
print(f"\nTotal registros: {len(df_perdif_check):,}")

print("\n=== EDADES CON PERDIF != 3 (no están en años) ===")
no_anios = df_perdif_check[df_perdif_check['perdif'] != 3.0]
print(f"Registros no en años: {len(no_anios):,} ({len(no_anios)/len(df_perdif_check)*100:.1f}%)")
print(f"\nDistribución de edad en esos registros:")
print(no_anios['edadif'].describe().round(1))

=== DISTRIBUCIÓN DE PERDIF (muestra 2022) ===
perdif
1.0     3387
2.0     2868
3.0    88888
9.0      243
Name: count, dtype: int64

Total registros: 95,386

=== EDADES CON PERDIF != 3 (no están en años) ===
Registros no en años: 6,498 (6.8%)

Distribución de edad en esos registros:
count    6498.0
mean       43.6
std       188.4
min         0.0
25%         2.0
50%         4.0
75%        10.0
max       999.0
Name: edadif, dtype: float64


In [41]:
# Recarga incluyendo perdif para corregir unidad de edad
COLS_DEF = ['depocu', 'mupocu', 'sexo', 'diaocu', 'mesocu', 'edadif', 'perdif', 'caudef']

frames = []
for año in range(2011, 2023):
    df_temp, _ = pyreadstat.read_sav(f"defunciones/def{año}.sav")
    df_temp.columns = df_temp.columns.str.lower()
    df_temp = df_temp.rename(columns={'añoocu': 'anio'})
    if 'anio' not in df_temp.columns:
        df_temp['anio'] = año
    cols_disponibles = [c for c in COLS_DEF if c in df_temp.columns]
    df_temp = df_temp[cols_disponibles + ['anio']].copy()
    frames.append(df_temp)

defunciones = pd.concat(frames, ignore_index=True)

# Convertir edad a años según perdif
def convertir_edad_anios(row):
    edad = row['edadif']
    perdif = row['perdif']
    if pd.isna(edad) or edad == 999:
        return np.nan
    if perdif == 1.0:   # días → años
        return edad / 365.25
    elif perdif == 2.0: # meses → años
        return edad / 12
    elif perdif == 3.0: # ya está en años
        return edad
    else:               # ignorado
        return np.nan

defunciones['edad'] = defunciones.apply(convertir_edad_anios, axis=1)

print("=== EDAD CORREGIDA EN DEFUNCIONES ===")
print(defunciones['edad'].describe().round(2))
print(f"\nNulos en edad tras corrección: {defunciones['edad'].isna().sum():,}")

=== EDAD CORREGIDA EN DEFUNCIONES ===
count    1016125.00
mean          54.64
std           28.51
min            0.00
25%           34.00
50%           62.00
75%           78.00
max          122.00
Name: edad, dtype: float64

Nulos en edad tras corrección: 7,022


In [42]:
# Clasificar defunciones
defunciones['es_muerte_natural'] = defunciones['caudef'].apply(clasificar_causa_def)
defunciones = defunciones[defunciones['es_muerte_natural'].notna()].reset_index(drop=True)

# Renombrar al esquema estándar
defunciones_final = defunciones.rename(columns={
    'mesocu': 'mes',
    'diaocu': 'dia',
    'depocu': 'depto',
}).copy()

COLS_FINALES = ['anio', 'mes', 'dia', 'depto', 'edad', 'sexo', 'es_muerte_natural']
defunciones_final = defunciones_final[COLS_FINALES]
necropsias_final  = necropsias[COLS_FINALES]

# Unir
df = pd.concat([necropsias_final, defunciones_final], ignore_index=True)

# Limpieza final
df['edad'] = df['edad'].replace(999.0, np.nan)
df['sexo'] = df['sexo'].replace(9.0, np.nan)
df['dia']  = df.groupby('mes')['dia'].transform(lambda x: x.fillna(x.median()))
df = df.dropna(subset=['edad', 'sexo']).reset_index(drop=True)

print("=== DATASET FINAL ===")
print(f"Total registros : {len(df):,}")
print(f"Columnas        : {list(df.columns)}")
print(f"\nNulos restantes:")
print(df.isnull().sum())
print(f"\nBalance de clases:")
vc = df['es_muerte_natural'].value_counts()
pct = df['es_muerte_natural'].value_counts(normalize=True).mul(100).round(1)
print(pd.DataFrame({'Conteo': vc, 'Porcentaje': pct}).to_string())
print(f"\nEstadísticas de edad corregida:")
print(df['edad'].describe().round(2))

=== DATASET FINAL ===
Total registros : 980,303
Columnas        : ['anio', 'mes', 'dia', 'depto', 'edad', 'sexo', 'es_muerte_natural']

Nulos restantes:
anio                 0
mes                  0
dia                  0
depto                0
edad                 0
sexo                 0
es_muerte_natural    0
dtype: int64

Balance de clases:
                   Conteo  Porcentaje
es_muerte_natural                    
1.0                887468        90.5
0.0                 92835         9.5

Estadísticas de edad corregida:
count    980303.00
mean         55.20
std          28.54
min           0.00
25%          35.00
50%          62.00
75%          78.00
max         122.00
Name: edad, dtype: float64


In [44]:
from sklearn.utils import resample

# Separar clases
df_natural     = df[df['es_muerte_natural'] == 1.0]
df_no_natural  = df[df['es_muerte_natural'] == 0.0]

print(f"Clase natural     (1): {len(df_natural):,}")
print(f"Clase no natural  (0): {len(df_no_natural):,}")

# Undersample de naturales al tamaño de no naturales
df_natural_under = resample(
    df_natural,
    replace=False,
    n_samples=len(df_no_natural),
    random_state=42
)

# Reunir y mezclar
df_balanceado = pd.concat([df_natural_under, df_no_natural]).sample(
    frac=1, random_state=42
).reset_index(drop=True)

print(f"\n=== DATASET BALANCEADO ===")
print(f"Total registros : {len(df_balanceado):,}")
print(f"\nBalance final:")
vc  = df_balanceado['es_muerte_natural'].value_counts()
pct = df_balanceado['es_muerte_natural'].value_counts(normalize=True).mul(100).round(1)
print(pd.DataFrame({'Conteo': vc, 'Porcentaje': pct}).to_string())
print(f"\nEstadísticas de edad:")
print(df_balanceado['edad'].describe().round(2))

Clase natural     (1): 887,468
Clase no natural  (0): 92,835

=== DATASET BALANCEADO ===
Total registros : 185,670

Balance final:
                   Conteo  Porcentaje
es_muerte_natural                    
0.0                 92835        50.0
1.0                 92835        50.0

Estadísticas de edad:
count    185670.00
mean         45.64
std          26.17
min           0.00
25%          25.00
50%          42.00
75%          68.00
max         118.00
Name: edad, dtype: float64


In [45]:
import os

# Crear carpeta si no existe
os.makedirs("data_bal", exist_ok=True)

# Guardar
df_balanceado.to_csv("data_bal/data_balanceada.csv", index=False)

print("Dataset guardado en data_bal/data_balanceada.csv")
print(f"Registros: {len(df_balanceado):,}")
print(f"Columnas : {list(df_balanceado.columns)}")

Dataset guardado en data_bal/data_balanceada.csv
Registros: 185,670
Columnas : ['anio', 'mes', 'dia', 'depto', 'edad', 'sexo', 'es_muerte_natural']
